# MAPPO Training — Fix 4

Multi-Agent PPO for UAV crop-disease monitoring.

**Key fixes over Fix 3:**
- Dynamic disease spread (runtime, not pre-baked)
- Eq. 11 potential-difference reward (replaces PBRS)
- Per-UAV GAE advantages (replaces pooled advantages)
- All reward shaping lives in `uav_env_4.py` — training loop is clean
- Balanced reward economy (discovery bonus ~8× per-step, not 200×)

## Setup

In [ ]:
import os, sys

ON_KAGGLE = os.path.exists('/kaggle/input')

if ON_KAGGLE:
    # Upload uav_env_4.py and networks_4.py as a dataset named 'uav-fix4'
    MODULE_DIR  = '/kaggle/input/uav-fix4'
    WEIGHTS_DIR = '/kaggle/working/weights_fix4'
    RESULTS_DIR = '/kaggle/working/results_fix4'
else:
    # Local: notebook lives in project/fixes-1/
    MODULE_DIR  = os.path.dirname(os.path.abspath('__file__'))
    ROOT_DIR    = os.path.dirname(os.path.dirname(MODULE_DIR))
    WEIGHTS_DIR = os.path.join(ROOT_DIR, 'weights_fix4')
    RESULTS_DIR = os.path.join(ROOT_DIR, 'results_fix4')

if MODULE_DIR not in sys.path:
    sys.path.insert(0, MODULE_DIR)

os.makedirs(WEIGHTS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
CKPT_DIR = os.path.join(WEIGHTS_DIR, 'checkpoints')
BEST_DIR = os.path.join(WEIGHTS_DIR, 'best')
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(BEST_DIR, exist_ok=True)

print('MODULE_DIR  :', MODULE_DIR)
print('WEIGHTS_DIR :', WEIGHTS_DIR)
print('RESULTS_DIR :', RESULTS_DIR)

In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from collections import deque
from tqdm.auto import tqdm

from uav_env_4 import (
    UAVFieldEnv, N_UAVS, N_SECTORS, OBS_SIZE, DAILY_STEPS_MAX,
    GRID_ROWS, GRID_COLS,
)
from networks_4 import SectorAttentionActor, CriticNetwork, JOINT_SIZE

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device     : {DEVICE}')
print(f'OBS_SIZE   : {OBS_SIZE}')
print(f'JOINT_SIZE : {JOINT_SIZE}')

## Hyperparameters

In [ ]:
N_EPISODES      = 400
K_EPOCHS        = 4
MINI_BATCH_SIZE = 128
GAMMA           = 0.99       # RL discount (distinct from env's GAMMA=0.8 history decay)
GAE_LAMBDA      = 0.95
EPS_CLIP        = 0.2
ENTROPY_COEFF   = 0.001
LR_ACTOR        = 3e-4
LR_CRITIC       = 3e-4
LR_MIN          = 1e-5
SAVE_INTERVAL   = 25
LOG_INTERVAL    = 5

## Environment & Model Initialisation

In [ ]:
env = UAVFieldEnv()   # no arguments — disease is generated dynamically each episode

actors = [SectorAttentionActor().to(DEVICE) for _ in range(N_UAVS)]
critic = CriticNetwork().to(DEVICE)

# Initialise log-std to -1  (std ≈ 0.37 — moderate exploration)
for actor in actors:
    nn.init.constant_(actor.action_log_std, -1.0)

actor_opts = [torch.optim.Adam(a.parameters(), lr=LR_ACTOR) for a in actors]
critic_opt  = torch.optim.Adam(critic.parameters(), lr=LR_CRITIC)

actor_scheds = [
    torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=N_EPISODES, eta_min=LR_MIN)
    for opt in actor_opts
]
critic_sched = torch.optim.lr_scheduler.CosineAnnealingLR(
    critic_opt, T_max=N_EPISODES, eta_min=LR_MIN
)

def count_params(m):
    return sum(p.numel() for p in m.parameters())

print(f'Actor params : {count_params(actors[0]):>10,}  ×{N_UAVS} = {count_params(actors[0]) * N_UAVS:,}')
print(f'Critic params: {count_params(critic):>10,}')
print(f'Total params : {count_params(actors[0]) * N_UAVS + count_params(critic):>10,}')

## Training Components

In [ ]:
class RunningMeanStd:
    """Online Welford mean/variance for reward normalisation."""

    def __init__(self, epsilon=1e-4):
        self.mean  = 0.0
        self.var   = 1.0
        self.count = epsilon

    def update(self, xs):
        batch = np.asarray(xs, dtype=np.float64).ravel()
        if batch.size == 0:
            return
        batch_mean = batch.mean()
        batch_var  = batch.var() if batch.size > 1 else 0.0
        delta      = batch_mean - self.mean
        total      = self.count + batch.size
        self.mean  = self.mean + delta * batch.size / total
        self.var   = (
            self.var * self.count
            + batch_var * batch.size
            + delta ** 2 * self.count * batch.size / total
        ) / total
        self.count = total

    def normalize(self, xs):
        return (np.asarray(xs, dtype=np.float32) - self.mean) / (np.sqrt(self.var) + 1e-8)

In [ ]:
class RolloutBuffer:
    def __init__(self):
        self.clear()

    def clear(self):
        self.obs       = [[] for _ in range(N_UAVS)]
        self.actions   = [[] for _ in range(N_UAVS)]
        self.log_probs = [[] for _ in range(N_UAVS)]
        self.rewards   = [[] for _ in range(N_UAVS)]
        self.values    = []
        self.dones     = []

    def store(self, obs_list, actions_list, log_probs_list, rewards_list, value, done):
        for u in range(N_UAVS):
            self.obs[u].append(obs_list[u])
            self.actions[u].append(actions_list[u])
            self.log_probs[u].append(log_probs_list[u])
            self.rewards[u].append(rewards_list[u])
        self.values.append(value)
        self.dones.append(done)

    def get_tensors(self):
        obs_t  = [
            torch.as_tensor(np.asarray(self.obs[u]),     dtype=torch.float32, device=DEVICE)
            for u in range(N_UAVS)
        ]
        acts_t = [
            torch.as_tensor(np.asarray(self.actions[u]), dtype=torch.float32, device=DEVICE)
            for u in range(N_UAVS)
        ]
        lps_t  = [
            torch.stack(self.log_probs[u]).squeeze(-1).to(DEVICE)
            for u in range(N_UAVS)
        ]
        rews_t = [
            torch.as_tensor(self.rewards[u], dtype=torch.float32, device=DEVICE)
            for u in range(N_UAVS)
        ]
        vals_t  = torch.stack(self.values).view(-1).to(DEVICE)
        dones_t = torch.as_tensor(self.dones, dtype=torch.float32, device=DEVICE)
        return obs_t, acts_t, lps_t, rews_t, vals_t, dones_t

In [ ]:
def compute_gae(rewards_list, values, dones, last_value):
    """
    Returns:
        per_uav_adv : list of N_UAVS tensors (T,) — per-UAV normalised advantages
        returns     : tensor (T,)               — shared critic targets (mean reward)

    Critic is trained on mean-team reward; each actor uses its own reward
    against the centralised baseline (CTDE pattern).
    """
    t_ep       = len(values)
    values_ext = torch.cat([values.view(-1), last_value.view(1)])

    # ── Shared returns for critic ──────────────────────────────────────────
    mean_rews  = torch.stack(rewards_list).mean(dim=0)
    shared_adv = torch.zeros(t_ep, device=DEVICE)
    gae        = 0.0
    for t in reversed(range(t_ep)):
        mask           = 1.0 - dones[t]
        delta          = mean_rews[t] + GAMMA * values_ext[t + 1] * mask - values_ext[t]
        gae            = delta + GAMMA * GAE_LAMBDA * mask * gae
        shared_adv[t]  = gae
    returns = shared_adv + values

    # ── Per-UAV advantages (individual reward, centralised baseline) ───────
    per_uav_adv = []
    for u in range(N_UAVS):
        adv_u  = torch.zeros(t_ep, device=DEVICE)
        gae_u  = 0.0
        for t in reversed(range(t_ep)):
            mask    = 1.0 - dones[t]
            delta   = rewards_list[u][t] + GAMMA * values_ext[t + 1] * mask - values_ext[t]
            gae_u   = delta + GAMMA * GAE_LAMBDA * mask * gae_u
            adv_u[t] = gae_u
        adv_u = (adv_u - adv_u.mean()) / (adv_u.std() + 1e-8)
        per_uav_adv.append(adv_u.detach())

    return per_uav_adv, returns.detach()

In [ ]:
def ppo_update(env, actors, critic, actor_opts, critic_opt, buffer):
    obs_t, acts_t, old_lps_t, rews_t, vals_t, dones_t = buffer.get_tensors()

    # Bootstrap value from terminal state
    with torch.no_grad():
        last_joint = torch.cat(
            [torch.as_tensor(env._get_obs(u), dtype=torch.float32, device=DEVICE)
             for u in range(N_UAVS)]
        ).unsqueeze(0)
        last_value = critic(last_joint).squeeze()

    per_uav_adv, returns = compute_gae(rews_t, vals_t, dones_t, last_value)

    t_ep          = vals_t.shape[0]
    actor_losses  = []
    critic_losses = []

    for _ in range(K_EPOCHS):
        perm = torch.randperm(t_ep, device=DEVICE)
        for start in range(0, t_ep, MINI_BATCH_SIZE):
            mb_idx     = perm[start : start + MINI_BATCH_SIZE]
            mb_returns = returns[mb_idx]

            # ── Actor updates (one per UAV) ────────────────────────────────
            for u in range(N_UAVS):
                mb_obs     = obs_t[u][mb_idx]
                mb_actions = acts_t[u][mb_idx]
                mb_old_lps = old_lps_t[u][mb_idx].detach()
                mb_adv     = per_uav_adv[u][mb_idx]

                new_lps, entropy = actors[u].get_log_prob_entropy(mb_obs, mb_actions)
                ratio   = torch.exp(new_lps - mb_old_lps)
                surr1   = ratio * mb_adv
                surr2   = torch.clamp(ratio, 1 - EPS_CLIP, 1 + EPS_CLIP) * mb_adv
                a_loss  = -torch.min(surr1, surr2).mean() - ENTROPY_COEFF * entropy

                actor_opts[u].zero_grad()
                a_loss.backward()
                nn.utils.clip_grad_norm_(actors[u].parameters(), 0.5)
                actor_opts[u].step()
                actor_losses.append(a_loss.item())

            # ── Critic update ─────────────────────────────────────────────
            mb_joint    = torch.cat([obs_t[u][mb_idx] for u in range(N_UAVS)], dim=1)
            values_pred = critic(mb_joint).squeeze()
            c_loss      = nn.MSELoss()(values_pred, mb_returns)

            critic_opt.zero_grad()
            c_loss.backward()
            nn.utils.clip_grad_norm_(critic.parameters(), 0.5)
            critic_opt.step()
            critic_losses.append(c_loss.item())

    return float(np.mean(actor_losses)), float(np.mean(critic_losses))

## Training Loop

In [ ]:
reward_rms      = RunningMeanStd()
buffer          = RolloutBuffer()
ep_rewards      = []
ep_diagnosed    = []
ep_inf_known    = []
ep_crashes      = []
ep_detect_rates = []   # fraction of ever-infected sectors the UAVs found
a_loss_log      = []
c_loss_log      = []
recent_rews     = deque(maxlen=50)
best_diag       = 0
best_det_rate   = 0.0
train_start     = time.time()

pbar = tqdm(range(1, N_EPISODES + 1), desc='Training', unit='ep')
for episode in pbar:
    obs            = env.reset()
    buffer.clear()
    ep_reward      = 0.0
    ep_crash_count = 0

    # ── Collect episode ───────────────────────────────────────────────────
    for _ in range(env.total_steps):
        joint_obs = torch.cat(
            [torch.as_tensor(obs[u], dtype=torch.float32, device=DEVICE)
             for u in range(N_UAVS)]
        ).unsqueeze(0)

        with torch.no_grad():
            value = critic(joint_obs)

        actions   = []
        log_probs = []
        for u in range(N_UAVS):
            obs_t = torch.as_tensor(obs[u], dtype=torch.float32, device=DEVICE)
            with torch.no_grad():
                action, lp = actors[u].get_action(obs_t)
            actions.append(action)
            log_probs.append(lp)

        next_obs, rewards, done, info = env.step(actions)
        ep_crash_count += sum(info['newly_crashed'])
        buffer.store(obs, actions, log_probs, rewards, value, float(done))
        ep_reward += float(sum(rewards))
        obs = next_obs
        if done:
            break

    # ── Normalise rewards in buffer ───────────────────────────────────────
    all_rews = np.concatenate(
        [np.asarray(buffer.rewards[u], dtype=np.float32) for u in range(N_UAVS)]
    )
    reward_rms.update(all_rews)
    for u in range(N_UAVS):
        buffer.rewards[u] = list(reward_rms.normalize(buffer.rewards[u]))

    # ── PPO update ────────────────────────────────────────────────────────
    a_loss, c_loss = ppo_update(env, actors, critic, actor_opts, critic_opt, buffer)

    for sched in actor_scheds:
        sched.step()
    critic_sched.step()

    # ── Metrics ───────────────────────────────────────────────────────────
    n_diag     = int((env.uav_status != 2).sum())
    n_inf      = int((env.uav_status == 1).sum())

    # Disease detection efficiency:
    #   ever_infected  = sectors that were infected at any point this episode
    #   detected       = those that UAVs actually diagnosed as infected
    #   detect_rate    = detected / ever_infected  (0.0–1.0)
    n_ever_inf  = int(env.ever_infected.sum())
    n_detected  = int((env.ever_infected & env.ever_diagnosed).sum())
    detect_rate = n_detected / max(n_ever_inf, 1)

    ep_rewards.append(ep_reward)
    ep_diagnosed.append(n_diag)
    ep_inf_known.append(n_inf)
    ep_crashes.append(ep_crash_count)
    ep_detect_rates.append(detect_rate)
    a_loss_log.append(a_loss)
    c_loss_log.append(c_loss)
    recent_rews.append(ep_reward)

    # ── Best checkpoint (by diagnosed count) ─────────────────────────────
    if n_diag > best_diag:
        best_diag = n_diag
        for u in range(N_UAVS):
            torch.save(actors[u].state_dict(),
                       os.path.join(BEST_DIR, f'actor{u}_best.pth'))
        torch.save(critic.state_dict(), os.path.join(BEST_DIR, 'critic_best.pth'))

    # ── Periodic checkpoint ───────────────────────────────────────────────
    if episode % SAVE_INTERVAL == 0:
        for u in range(N_UAVS):
            torch.save(actors[u].state_dict(),
                       os.path.join(CKPT_DIR, f'actor{u}_ep{episode}.pth'))
        torch.save(critic.state_dict(),
                   os.path.join(CKPT_DIR, f'critic_ep{episode}.pth'))

    # ── Progress bar ──────────────────────────────────────────────────────
    avg50 = float(np.mean(recent_rews))
    pbar.set_postfix({
        'rew':    f'{ep_reward:.1f}',
        'avg50':  f'{avg50:.1f}',
        'diag':   f'{n_diag}/{N_SECTORS}',
        'det%':   f'{detect_rate*100:.0f}%',
        'crash':  ep_crash_count,
        'a_loss': f'{a_loss:.4f}',
    })

    if episode % LOG_INTERVAL == 0 or episode == 1:
        elapsed  = time.time() - train_start
        eta_min  = elapsed / episode * (N_EPISODES - episode) / 60
        tqdm.write(
            f'Ep {episode:>4}/{N_EPISODES}  '
            f'rew={ep_reward:>9.2f}  avg50={avg50:>9.2f}  '
            f'diag={n_diag:>3}/{N_SECTORS}  '
            f'det={detect_rate*100:5.1f}%  ({n_detected}/{n_ever_inf})  '
            f'crash={ep_crash_count:>3}  '
            f'a={a_loss:.4f}  c={c_loss:.4f}  '
            f'lr={actor_opts[0].param_groups[0]["lr"]:.2e}  ETA={eta_min:.1f}min'
        )

total_time = time.time() - train_start
print(f'\nTraining complete in {total_time/60:.1f} min')
print(f'Best diagnosed  : {best_diag} / {N_SECTORS}')
print(f'Best detect rate: {max(ep_detect_rates)*100:.1f}%')
print(f'Total crashes   : {sum(ep_crashes)}')

## Training Results

In [ ]:
def moving_avg(data, w=50):
    return np.convolve(data, np.ones(w) / w, mode='valid')

fig, axes = plt.subplots(2, 3, figsize=(18, 8))
fig.suptitle('MAPPO Training — Fix 4', fontsize=13)

# ── Row 0 ─────────────────────────────────────────────────────────────────────

axes[0, 0].plot(ep_rewards, alpha=0.2, color='steelblue')
if len(ep_rewards) >= 50:
    axes[0, 0].plot(moving_avg(ep_rewards), color='steelblue', label='MA-50')
    axes[0, 0].legend()
axes[0, 0].set_title('Episode Reward')
axes[0, 0].set_xlabel('Episode')
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(ep_diagnosed, alpha=0.2, color='tomato')
if len(ep_diagnosed) >= 50:
    axes[0, 1].plot(moving_avg(ep_diagnosed), color='tomato', label='MA-50')
    axes[0, 1].legend()
axes[0, 1].axhline(N_SECTORS, color='k', ls='--', lw=0.8)
axes[0, 1].set_title('Total Sectors Diagnosed / Episode')
axes[0, 1].set_xlabel('Episode')
axes[0, 1].grid(True, alpha=0.3)

axes[0, 2].plot(ep_crashes, alpha=0.2, color='purple')
if len(ep_crashes) >= 50:
    axes[0, 2].plot(moving_avg(ep_crashes), color='purple', label='MA-50')
    axes[0, 2].legend()
axes[0, 2].axhline(0, color='k', ls='--', lw=0.8)
axes[0, 2].set_title('Crashes / Episode  (target → 0)')
axes[0, 2].set_xlabel('Episode')
axes[0, 2].grid(True, alpha=0.3)

# ── Row 1 ─────────────────────────────────────────────────────────────────────

pct = [r * 100 for r in ep_detect_rates]
axes[1, 0].plot(pct, alpha=0.2, color='darkorange')
if len(pct) >= 50:
    axes[1, 0].plot(moving_avg(pct), color='darkorange', label='MA-50')
    axes[1, 0].legend()
axes[1, 0].axhline(100, color='k', ls='--', lw=0.8)
axes[1, 0].set_ylim(0, 105)
axes[1, 0].set_title('Disease Detection Rate  (%infected found)')
axes[1, 0].set_xlabel('Episode')
axes[1, 0].set_ylabel('%')
axes[1, 0].grid(True, alpha=0.3)

if len(a_loss_log) >= 50:
    axes[1, 1].plot(moving_avg(a_loss_log), color='firebrick')
axes[1, 1].set_title('Actor Loss (MA-50)')
axes[1, 1].set_xlabel('Episode')
axes[1, 1].grid(True, alpha=0.3)

if len(c_loss_log) >= 50:
    axes[1, 2].plot(moving_avg(c_loss_log), color='mediumseagreen')
axes[1, 2].set_title('Critic Loss (MA-50)')
axes[1, 2].set_xlabel('Episode')
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
out_path = os.path.join(RESULTS_DIR, 'training_curves_fix4.png')
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print('Saved:', out_path)

## Save Final Weights

In [ ]:
for u in range(N_UAVS):
    path = os.path.join(WEIGHTS_DIR, f'actor{u}_final.pth')
    torch.save(actors[u].state_dict(), path)
    print(f'Saved actor {u} → {path}')

path = os.path.join(WEIGHTS_DIR, 'critic_final.pth')
torch.save(critic.state_dict(), path)
print(f'Saved critic  → {path}')
print(f'Best weights  → {BEST_DIR}/')

## Evaluation

Loads the best-checkpoint weights and runs a single deterministic episode
(uses `dist.loc` — the mean action — instead of sampling).

In [ ]:
eval_actors = [SectorAttentionActor().to(DEVICE) for _ in range(N_UAVS)]
for u in range(N_UAVS):
    path = os.path.join(BEST_DIR, f'actor{u}_best.pth')
    eval_actors[u].load_state_dict(torch.load(path, map_location=DEVICE))
    eval_actors[u].eval()
    print(f'Loaded actor {u} from {path}')

eval_env  = UAVFieldEnv(seed=0)   # fixed seed for reproducible eval
obs       = eval_env.reset()
eval_hist = []
ep_rew    = 0.0

for _ in range(eval_env.total_steps):
    actions = []
    for u in range(N_UAVS):
        obs_t = torch.as_tensor(obs[u], dtype=torch.float32, device=DEVICE).unsqueeze(0)
        with torch.no_grad():
            action = eval_actors[u](obs_t).loc.squeeze(0).cpu().numpy()
        actions.append(action)

    obs, rewards, done, info = eval_env.step(actions)
    ep_rew += sum(rewards)
    eval_hist.append(info)
    if done:
        break

print(f'Eval episode reward : {ep_rew:.2f}')

In [ ]:
timesteps   = [s['t'] for s in eval_hist]
n_visited   = [(s['uav_status'] != 2).sum() for s in eval_hist]
n_infected  = [s['uav_status'].sum() == 1 and (s['uav_status'] == 1).sum() for s in eval_hist]
n_inf_known = [(s['uav_status'] == 1).sum() for s in eval_hist]
n_true_inf  = [s['true_status'].sum() for s in eval_hist]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Evaluation Episode (deterministic policy)', fontsize=12)

axes[0].plot(timesteps, n_visited, color='steelblue')
axes[0].set_title('Sectors Diagnosed'); axes[0].set_xlabel('Step')
axes[0].axhline(N_SECTORS, color='k', ls='--', lw=0.8); axes[0].grid(True, alpha=0.3)

axes[1].plot(timesteps, n_true_inf, color='tomato', label='True infected')
axes[1].plot(timesteps, n_inf_known, color='orange', ls='--', label='UAV-known infected')
axes[1].set_title('Infection Tracking'); axes[1].set_xlabel('Step')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(timesteps,
             [s['energy'][0] for s in eval_hist], label='UAV 0')
axes[2].plot(timesteps,
             [s['energy'][1] for s in eval_hist], label='UAV 1')
axes[2].plot(timesteps,
             [s['energy'][2] for s in eval_hist], label='UAV 2')
axes[2].plot(timesteps,
             [s['energy'][3] for s in eval_hist], label='UAV 3')
axes[2].set_title('UAV Energy'); axes[2].set_xlabel('Step')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
out_path = os.path.join(RESULTS_DIR, 'eval_fix4.png')
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print('Saved:', out_path)

In [ ]:
final   = eval_hist[-1]
n_diag  = int((final['uav_status'] != 2).sum())
n_inf_k = int((final['uav_status'] == 1).sum())
n_true  = int(final['true_status'].sum())
n_crash = sum(1 for h in eval_hist if any(h['newly_crashed']))

print('=' * 55)
print('  EVALUATION SUMMARY  (Fix 4)')
print('=' * 55)
print(f'  Grid             : {GRID_ROWS}×{GRID_COLS} = {N_SECTORS} sectors')
print(f'  Episode reward   : {ep_rew:.2f}')
print(f'  Sectors diagnosed: {n_diag} / {N_SECTORS}  ({100*n_diag/N_SECTORS:.1f}%)')
print(f'  UAV-known infected: {n_inf_k}')
print(f'  True infected     : {n_true}')
print(f'  Crash steps       : {n_crash}')
print(f'  Best train diag   : {best_diag} / {N_SECTORS}')
print('=' * 55)